## Ch8-04 -- A real requirement ties the lemma

This notebook introduces `requirement def EnergyConservationReq` and its usage `energyConservationReq`; after running it you can show `deliveredEnergyBoundedBySupply` is now tied to a real requirement, where before it was tied to none.

[Ch8-01](01-invariant-def.ipynb) and [Ch8-02](02-violation-witness.ipynb) stated and proved `deliveredEnergyBoundedBySupply`, but tied it to no requirement: Chapter 10's own traceability search (`01-traceability-graph.ipynb` cell 18, `03-engineering-signoff.ipynb` cell 7; `decisions/log.md` DL-070, DL-071) correctly and honestly reported it as an "unjustified widget in reverse" -- a proved property nothing actually requires. This notebook adds `EnergyConservationReq`, a real `requirement def` whose own `require constraint` subsets the lemma directly, closing that gap for this specific construct. [Ch8-05](05-verification-case.ipynb) adds the matching `verification def` that closes the requirement's own anatomy, mirroring `TimelyToast`/`TimelyToastTest` (`models/ch08-cumulative.sysml` lines 76-108).

In [1]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")

# editor.add_requirement_def(owner='ToasterDemo', name='EnergyConservationReq', doc=..., subject_type='HeatGenerator') when API ships
# spec: SysML v2 formal/2026-03-02 SS7.21.2: doc gives the informal text (description + rationale)
ENERGY_REQ_DEF = """\
requirement def EnergyConservationReq {
    doc /*
     * A heat generator's delivered energy shall never exceed its supplied
     * energy. Rationale: no mechanism can output more thermal energy than
     * the electrical energy supplied to it; this is the conservation
     * relation deliveredEnergyBoundedBySupply already proves, by Z3, for
     * every value its unbound features admit (Ch8-01, Ch8-02). The
     * require constraint below ties this requirement directly to that
     * proved lemma by subsetting it (:>), rather than restating the
     * lemma's own body a second time.
     */
    subject heatGen : HeatGenerator;
"""
print(ENERGY_REQ_DEF)

requirement def EnergyConservationReq {
    doc /*
     * A heat generator's delivered energy shall never exceed its supplied
     * energy. Rationale: no mechanism can output more thermal energy than
     * the electrical energy supplied to it; this is the conservation
     * relation deliveredEnergyBoundedBySupply already proves, by Z3, for
     * every value its unbound features admit (Ch8-01, Ch8-02). The
     * require constraint below ties this requirement directly to that
     * proved lemma by subsetting it (:>), rather than restating the
     * lemma's own body a second time.
     */
    subject heatGen : HeatGenerator;



`EnergyConservationReq` is a requirement definition, the same construct [Ch2-01](../ch02-requirements/01-requirement-def.ipynb) introduced for `TimelyToast`. Its `doc` states the need and the rationale; its `subject heatGen : HeatGenerator` names what the requirement is about -- any heat generator, not one specific candidate.

In [2]:
# editor.add_require_constraint(owner='ToasterDemo::EnergyConservationReq', subsets=['deliveredEnergyBoundedBySupply']) when API ships
# require constraint: the Editor API does not yet author it (toaster#11 / OpenSysML#597);
# it parses and loads correctly via conn.load_from_content(), confirmed below.
# spec: SysML v2 formal/2026-03-02 SS7.19 (RequirementConstraintMembership); SS8.3 (subsetting, :>)
ENERGY_CONSTRAINT_TIE = "    require constraint energyConserved :> deliveredEnergyBoundedBySupply;"
print(ENERGY_CONSTRAINT_TIE)

    require constraint energyConserved :> deliveredEnergyBoundedBySupply;


`energyConserved` is a new `require constraint` that subsets (`:>`) `deliveredEnergyBoundedBySupply` directly, rather than restating the lemma's own body a second time. This is the tie itself: the requirement's own body now directly references the proved construct, which is exactly the shape `decisions/next-passes.md` item 29's own resolution confirms this toolchain finds (`decisions/log.md` DL-071's "RESOLVED, FINAL" positive control).

In [3]:
# editor.add_requirement(owner='ToasterDemo', name='energyConservationReq', type='EnergyConservationReq') when API ships
ENERGY_USAGE = "requirement energyConservationReq : EnergyConservationReq;"
print(ENERGY_USAGE)

requirement energyConservationReq : EnergyConservationReq;


`energyConservationReq` applies `EnergyConservationReq` the same way [Ch3-01](../ch03-measures/01-moe-definition.ipynb) applied `TimelyToast` as `timely`: a requirement definition describes what must hold, and a requirement usage applies it.

In [4]:
TOASTER_INCREMENT = f"{ENERGY_REQ_DEF}{ENERGY_CONSTRAINT_TIE}\n}}\n{ENERGY_USAGE}\n"
print(TOASTER_INCREMENT)

source = Path("../../models/ch08-cumulative.sysml").read_text()
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

requirement def EnergyConservationReq {
    doc /*
     * A heat generator's delivered energy shall never exceed its supplied
     * energy. Rationale: no mechanism can output more thermal energy than
     * the electrical energy supplied to it; this is the conservation
     * relation deliveredEnergyBoundedBySupply already proves, by Z3, for
     * every value its unbound features admit (Ch8-01, Ch8-02). The
     * require constraint below ties this requirement directly to that
     * proved lemma by subsetting it (:>), rather than restating the
     * lemma's own body a second time.
     */
    subject heatGen : HeatGenerator;
    require constraint energyConserved :> deliveredEnergyBoundedBySupply;
}
requirement energyConservationReq : EnergyConservationReq;



`ch08-cumulative.sysml` already carries this construct forward, the same as every other construction-zone cell in this tutorial: it is not assembled from the fragments above at runtime, it is the chapter's own committed fixture. The next cell checks what happens when a `require constraint` subsets a name that does not exist.

In [5]:
# Negative control: a require constraint that subsets an undefined name fails to load,
# the same "unresolved reference" failure mode as an undeclared attribute (Ch2-01, Ch8-01).
bad_source = """
package Bad {
    private import ScalarValues::*;
    part def Thing { attribute x : Real default = 5.0; }
    requirement def BadReq {
        subject t : Thing;
        require constraint c :> nonExistentConstraint;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok, "Expected failure for undefined subsetting target"
print(f"Negative control ok: bad.ok={bad.ok}")

Negative control ok: bad.ok=False


With the negative control confirmed, the next cell looks up `EnergyConservationReq` and `energyConservationReq` in the loaded model directly.

In [6]:
req_def = model.find("ToasterDemo::EnergyConservationReq")
assert req_def is not None
req_usage = model.find("ToasterDemo::energyConservationReq")
assert req_usage is not None
print(f"requirement def kind  : {req_def.kind}")
print(f"requirement usage kind: {req_usage.kind}")

NAMES = {"ToasterDemo::EnergyConservationReq", "ToasterDemo::energyConservationReq"}
for e in model.query():
    d = e.as_dict()
    if d.get("qualifiedName") in NAMES:
        print(f"{d['@type']}: {d['qualifiedName']}")

requirement def kind  : requirementDef
requirement usage kind: requirementUsage
RequirementDefinition: ToasterDemo::EnergyConservationReq
RequirementUsage: ToasterDemo::energyConservationReq


`model.find()` and `model.query()` confirm both elements are really part of the model. What they do not yet show is whether `deliveredEnergyBoundedBySupply` is now actually reachable from a real requirement -- `decisions/log.md` DL-070/DL-071 built exactly this check, and Chapter 10's own notebooks (cited above) ran it against the pre-existing model and got `False`. The next cell runs that same check against the model as loaded here.

In [7]:
from toaster import query

idx = query.ApiIndex(model)
LEMMA = "ToasterDemo::deliveredEnergyBoundedBySupply"
ties = query.requirement_ties(model, LEMMA, idx)
print("requirement_ties:", ties)

now_tied = query.tied_to_any_requirement(model, LEMMA, idx)
print(f"tied_to_any_requirement: {now_tied}")
assert now_tied is True, "Expected the lemma to now be tied via EnergyConservationReq"
conn.close()

requirement_ties: [{'tying_element': 'ToasterDemo::EnergyConservationReq::energyConserved', 'field': 'subsets', 'requirement': 'ToasterDemo::EnergyConservationReq'}]
tied_to_any_requirement: True


`tied_to_any_requirement` now returns `True`, where Chapter 10's own search against the pre-existing model returned `False` for this same lemma and this same target string: `energyConservationReq`'s own `require constraint energyConserved :> deliveredEnergyBoundedBySupply;` is found by Check B, attributed to `EnergyConservationReq`, exactly as printed above.

The requirement definition and usage printed above loaded without error, and `tied_to_any_requirement()` confirms the lemma is now actually reachable from a real requirement's own body, not merely declared alongside one.

Try the chapter exercise in `exercises/ch08/exercise.ipynb`: it works through the same `verify_satisfaction()` and stale-detection pattern on its own, separate coffee-maker exercise model.